In [0]:
pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers pypdf openai langchain-openai python-dotenv

In [0]:
# pip install langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load PDF
pdf_files = ["/Workspace/Users/surbhiwahie@gmail.com/Rag-System---Ask-My-Doc/Data/raw/Diabetes Care Booklet.pdf"]
all_docs = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    
    # Add source info
    for d in docs:
        d.metadata["source"] = file.split("/")[-1]
    
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages total")

In [0]:
# 2. Chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(all_docs)

print(f"Created {len(chunks)} chunks")

In [0]:
import shutil
shutil.rmtree("/tmp/chroma_db", ignore_errors=True)

In [0]:
# 3. Add metadata
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i

# 4. Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 5. Store in Chroma
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="/tmp/chroma_db"
)
db.persist()

print("✅ Vector DB created!")

In [0]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 2. Load DB
db = Chroma(
    persist_directory="/tmp/chroma_db",
    embedding_function=embedding_model
)

# 3. Retriever
retriever = db.as_retriever(search_kwargs={"k": 3})

# 4. Query
query = "symptoms of diabetes"
results = retriever.invoke(query)

# 5. Print
for r in results:
    print("----")
    print(r.page_content[:300])
    print(r.metadata)

In [0]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [0]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",   # cheaper + fast (good for your project)
    temperature=0,         # important → reduces hallucination
    api_key=api_key
)

In [0]:
def answer_question(query, retriever):
    
    # 1. Retrieve relevant chunks
    docs = retriever.invoke(query)
    
    # 2. Build context
    context = "\n\n".join([
        f"{d.page_content}\nSource: {d.metadata.get('source')}, Page: {d.metadata.get('page')}"
        for d in docs
    ])
    
    # 3. Strict prompt (VERY IMPORTANT)
    prompt = f"""
You are a strict AI assistant.

ONLY answer using the provided context.
If the answer is not in the context, say:
"I cannot find sufficient information in the provided documents."

Always include citations in this format:
[Source: filename, Page X]

Context:
{context}

Question:
{query}

Answer:
"""
    
    # 4. Call LLM
    response = llm.invoke(prompt)
    
    return response.content

In [0]:
# query = "What are the symptoms of diabetes?"
# query = "What is insulin?"
# query = "How is diabetes diagnosed?"
# query = "What is cancer?"
# query = "Does physical activity help my diabetes?"
query = "what is blood sugar?"

answer = answer_question(query, retriever)

print(answer)